# Signal Analysis

Analyzes the **statistical-arbitrage SPACES** (`research/lib/spaces.py`) from the
persisted research stats — `signal_metrics` (whole-period IC/turnover) and
`signal_daily_stats` (daily IC aggregates). Each space is **one economic hypothesis**
(relative cross-sectional displacement in space S predicts idiosyncratic returns);
`walk_forward` selects which have live edge each window.

Raw per-bar signal values are NOT stored — they are recomputed on demand from
`features` via `compute_signal_panel`. This notebook works from the compact stats:

- IC / ICIR by horizon (decay across 10min → 2d)
- IC / ICIR distributions
- **Theme** comparison (which spaces carry edge)
- Top spaces, IC-over-time, cumulative IC
- Edge vs turnover (the cost/horizon tradeoff)

> Set `signals.source = 'generator'` in config to analyze the legacy ~6k-signal
> universe instead. Numbers below reflect whatever the last `evaluate.py` run persisted.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

from dbutil import load_data, _connection
from research.signals.evaluate import build_registry

HORIZON_ORDER = ['1b', '3b', '6b', '18b', '36b', '72b', '144b', '288b']

# Whole-period diagnostics for every (signal_name, horizon)
metrics = load_data('signal_metrics')
metrics = metrics[metrics['error'].fillna('') == ''].copy()   # drop errored signals

# Theme + economic rationale per signal (from the active registry; spaces carry
# a one-line rationale, which is the interpretable payoff of the spaces design).
reg = build_registry()
fam = {name: info.get('category', name) for name, info in reg.items()}
rat = {name: info.get('description', '') for name, info in reg.items()}
metrics['theme'] = metrics['signal_name'].map(fam).fillna('unknown')
metrics['rationale'] = metrics['signal_name'].map(rat).fillna('')
metrics['horizon'] = pd.Categorical(metrics['horizon'], categories=HORIZON_ORDER, ordered=True)
metrics['ic_abs'] = metrics['ic_mean'].abs()
# `family` kept as an alias so the rest of the notebook reads either vocabulary.
metrics['family'] = metrics['theme']

print(f"{metrics['signal_name'].nunique():,} signals x {metrics['horizon'].nunique()} horizons "
      f"= {len(metrics):,} rows across {metrics['theme'].nunique()} themes")
metrics.head()

## The space library

Every space is one economic hypothesis with a named rationale — the interpretable
payoff of the spaces design (vs `delta_36__ms_trade_intensity_zscore`).

In [ ]:
# The active space library - one economic hypothesis per row (from the registry,
# so this works even before evaluate.py has been run). This is the interpretable
# catalog the spaces design buys you: every signal has a named rationale.
catalog = pd.DataFrame([
    {'space': name, 'theme': info.get('category', ''),
     'columns': ','.join(getattr(info['signal_def'], 'columns', ()) or []),
     'rationale': info.get('description', '')}
    for name, info in reg.items()
]).sort_values(['theme', 'space']).reset_index(drop=True)
print(f"{len(catalog)} spaces across {catalog['theme'].nunique()} themes")
display(catalog)

## IC by horizon

How predictive power varies across holding horizons (1 bar = 10 min ... 288 bars = 2 days).

In [ ]:
by_h = metrics.groupby('horizon', observed=True).agg(
    n_signals=('signal_name', 'nunique'),
    ic_abs_mean=('ic_abs', 'mean'),
    ic_abs_med=('ic_abs', 'median'),
    icir_abs_mean=('icir', lambda s: s.abs().mean()),
    turnover_mean=('avg_turnover_per_rebalance', 'mean'),
).reset_index()
print(by_h.round(4).to_string(index=False))

fig = px.bar(by_h, x='horizon', y='ic_abs_mean', title='Mean |IC| by horizon',
             labels={'ic_abs_mean': 'mean |IC|'})
fig.show()

## IC / ICIR distributions

Across all signals. Most cross-sectional signals have tiny IC (~0.01); the tail is what matters.

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=['IC mean (all signal-horizons)', 'ICIR (clipped +/-5)'])
fig.add_trace(go.Histogram(x=metrics['ic_mean'], nbinsx=80, showlegend=False), row=1, col=1)
fig.add_trace(go.Histogram(x=metrics['icir'].clip(-5, 5), nbinsx=80, showlegend=False), row=1, col=2)
fig.add_vline(x=0, line_dash='dash', line_color='gray', row=1, col=1)
fig.add_vline(x=0, line_dash='dash', line_color='gray', row=1, col=2)
fig.update_layout(height=400, title='Signal IC / ICIR distributions')
fig.show()

print(metrics[['ic_mean', 'icir', 'ic_tstat', 'avg_turnover_per_rebalance']].describe().round(4).to_string())

## Family comparison

Which families carry edge, ranked by mean |ICIR| (IC per unit IC-volatility).

In [ ]:
by_fam = metrics.groupby('family').agg(
    n=('signal_name', 'nunique'),
    ic_abs_mean=('ic_abs', 'mean'),
    icir_abs_mean=('icir', lambda s: s.abs().mean()),
    best_ic=('ic_abs', 'max'),
).reset_index().sort_values('icir_abs_mean', ascending=False)
print(by_fam.round(4).to_string(index=False))

fig = px.bar(by_fam.head(25), x='family', y='icir_abs_mean',
             title='Mean |ICIR| by family (top 25)', labels={'icir_abs_mean': 'mean |ICIR|'})
fig.update_xaxes(tickangle=45)
fig.show()

## Top signals

Ranked by |ICIR| across all signal-horizons.

In [ ]:
TOP_N = 25
top = metrics.reindex(metrics['icir'].abs().sort_values(ascending=False).index).head(TOP_N)
print(top[['signal_name', 'family', 'horizon', 'ic_mean', 'icir', 'ic_tstat',
           'avg_turnover_per_rebalance']].round(4).to_string(index=False))

## IC over time

Rebuilds the daily cross-sectional IC (`ic_sum / n_cs`) from `signal_daily_stats` for the
top signals at one horizon. Aggregated/filtered in DuckDB so we never load the 42M-row table.

In [ ]:
HZ = '6b'   # horizon to inspect over time
sel = (metrics[metrics['horizon'] == HZ]
       .reindex(metrics[metrics['horizon'] == HZ]['icir'].abs().sort_values(ascending=False).index)
       ['signal_name'].head(6).tolist())

ph = ','.join(['?'] * len(sel))
with _connection(read_only=True) as c:
    ds = c.execute(
        f"SELECT date, signal_name, ic_sum, n_cs FROM signal_daily_stats "
        f"WHERE signal_name IN ({ph}) AND horizon = ?", sel + [HZ]).fetchdf()
ds['date'] = pd.to_datetime(ds['date'])
ds['ic'] = ds['ic_sum'] / ds['n_cs'].replace(0, np.nan)
ds = ds.sort_values('date')

fig = go.Figure()
for sig, g in ds.groupby('signal_name'):
    fig.add_trace(go.Scatter(x=g['date'], y=g['ic'].rolling(20).mean(), name=sig, mode='lines'))
fig.add_hline(y=0, line_dash='dash', line_color='gray')
fig.update_layout(title=f'Rolling daily IC (20d) - top signals @ {HZ}',
                  yaxis_title='IC', xaxis_title='date', hovermode='x unified')
fig.show()

## Cumulative IC

Cumulative daily IC for the same signals - a steady climb means persistent edge, a flat
stretch means decay.

In [ ]:
fig = go.Figure()
for sig, g in ds.groupby('signal_name'):
    fig.add_trace(go.Scatter(x=g['date'], y=g['ic'].fillna(0).cumsum(), name=sig, mode='lines'))
fig.update_layout(title=f'Cumulative daily IC - top signals @ {HZ}',
                  yaxis_title='cumulative IC', xaxis_title='date', hovermode='x unified')
fig.show()

## Edge vs turnover

The cost/horizon tradeoff: high-IC signals that also turn over fast are expensive to trade.

In [ ]:
m = metrics[metrics['ic_abs'] < 0.3]   # clip the ml outliers for legibility
fig = px.scatter(m, x='avg_turnover_per_rebalance', y='ic_abs', color='horizon',
                 opacity=0.35, title='Edge vs turnover (per signal-horizon)',
                 labels={'ic_abs': '|IC mean|', 'avg_turnover_per_rebalance': 'turnover / rebalance'})
fig.show()

## Summary

In [ ]:
print('=' * 70)
print('SIGNAL ANALYSIS SUMMARY (persisted stats)')
print('=' * 70)

idx = metrics.dropna(subset=['icir']).groupby('family')['icir'].apply(lambda s: s.abs().idxmax())
best = metrics.loc[idx.values].sort_values('icir', key=lambda s: s.abs(), ascending=False)
print('Best signal per family (by |ICIR|):')
print(best[['family', 'signal_name', 'horizon', 'ic_mean', 'icir', 'ic_tstat',
            'avg_turnover_per_rebalance']].round(4).to_string(index=False))

print()
print(f"Universe: {metrics['signal_name'].nunique():,} signals x {metrics['family'].nunique()} "
      f"families x {len(HORIZON_ORDER)} horizons")
print()
print('Notes:')
print('- IC > ~0.01 is typical for these 10min-2d cross-sectional signals.')
print('- ic_tstat here is the pooled iid t-stat from signal_research; the walk-forward')
print('  selector uses a Newey-West HAC t-stat, which is smaller (and what gates entry).')
print('- Raw per-bar signal values are not stored; recompute via compute_signal_panel if needed.')